In [2]:
import os
import requests
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnablePassthrough

from dotenv import load_dotenv
load_dotenv("../.env")


True

In [3]:
## Start by creating an instance of the AzureChatOpenAI class.
llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    temperature=0,
)

In [15]:
#Define web search
@tool
def web_search(query: str) -> str:
    """Search the web for real-time or current information."""
    url = "https://serpapi.com/search.json"
    params = {
        "q": query,
        "api_key": os.getenv("SERPAPI_API_KEY"),
        "num": 5,
    }
    
    response = requests.get(url, params=params)
    data = response.json()

    results = data.get("organic_results", [])
    if not results:
        return "No results found."

    return "\n".join(
        r.get("snippet", "") for r in results[:3] if r.get("snippet")
    )

In [16]:
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(
        content=(
            "You are an intelligent assistant.\n"
            "If the question requires real-time or current information, "
            "call the appropriate tool.\n"
            "Otherwise answer directly."
        )
    ),
    ("human", "{input}")
])

In [17]:
#Bind tools to the model
llm_with_tools = llm.bind_tools([web_search])

In [18]:
#Build runnable chain
chain = (
    {"input": RunnablePassthrough()}
    | prompt
    | llm_with_tools
)

In [19]:
#Real time tool call
response = chain.invoke("What is the position of the moon relative to Germany right now?")
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 85, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 17, 'engine_ttlt_ms': 138, 'pre_inference_ms': 159, 'service_tbt_ms': 6, 'service_ttft_ms': 231, 'service_ttlt_ms': 349, 'total_duration_ms': 196, 'user_visible_ttft_ms': 72}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-De5Nw1ZyJ6ygKfJiab4KVVkboGBEu', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'seve

In [20]:
#To execute tool calls automatically
from langchain_core.messages import ToolMessage

def run_with_tools(chain, user_input):
    result = chain.invoke(user_input)

    if result.tool_calls:
        tool_outputs = []
        for call in result.tool_calls:
            tool_result = web_search.invoke(call["args"])
            tool_outputs.append(
                ToolMessage(
                    content=tool_result,
                    tool_call_id=call["id"]
                )
            )

        final = llm.invoke([result, *tool_outputs])
        return final.content

    return result.content

In [21]:
#running tools
run_with_tools(
    chain,
    "What is the position of the moon relative to Germany right now?"
)

'As of the current time on May 9, 2026, the Moon is in its Third Quarter phase. It is positioned roughly at 109.84° ESE (East-Southeast) relative to Germany, with an altitude of about -5.68°, meaning it is just below the horizon at the moment. The Moon is approximately 244,012 miles away from Earth.\n\nIf you want more detailed or real-time tracking, I can help with that as well.'

In [22]:
run_with_tools(
    chain,
    "Explain quantum entanglement simply."
)

'Quantum entanglement is a phenomenon in quantum physics where two or more particles become connected in such a way that the state of one particle instantly influences the state of the other, no matter how far apart they are. Imagine you have two magic coins that always show the same side when flipped, even if one coin is on Earth and the other is on the Moon. When you look at one coin and see it’s heads, you immediately know the other coin is also heads. This connection happens instantly, faster than anything can travel between them, which is what makes entanglement so strange and fascinating. It’s a key concept in quantum mechanics and has important implications for quantum computing and secure communication.'